# 🌟 NayePankh Foundation — Fundraising Amount Predictor
### Machine Learning Internship | Option 2 — ML Project

**Objective:** Build a machine learning model to predict the amount of funds a campaign can raise, based on campaign features, donor behavior, and seasonal patterns.

**Models Used:**
- Linear Regression (Baseline)
- Random Forest Regressor
- Gradient Boosting Regressor

**Dataset:** Synthetic but realistic dataset modeled after NGO fundraising patterns.


In [ ]:
# ============================================================
# 📦 IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print("✅ All libraries imported successfully!")


## 📊 Step 1: Dataset Creation
We generate a realistic synthetic dataset of **500 NGO fundraising campaigns** with features that an NGO like NayePankh Foundation would realistically track.


In [ ]:
# ============================================================
# 🗃️ GENERATE SYNTHETIC DATASET
# ============================================================
np.random.seed(42)
n = 500

campaign_types   = ['Education', 'Health', 'Women Empowerment', 'Environment', 'Child Welfare']
platforms        = ['Instagram', 'Facebook', 'WhatsApp', 'Email', 'Website']
months           = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
festive_months   = ['Oct', 'Nov', 'Dec', 'Jan']  # Higher giving during festive/year-end

data = {
    'campaign_type'        : np.random.choice(campaign_types, n),
    'platform'             : np.random.choice(platforms, n),
    'month'                : np.random.choice(months, n),
    'duration_days'        : np.random.randint(5, 60, n),
    'num_posts'            : np.random.randint(1, 50, n),
    'avg_post_reach'       : np.random.randint(200, 10000, n),
    'num_volunteers'       : np.random.randint(1, 80, n),
    'past_donors_count'    : np.random.randint(10, 500, n),
    'avg_past_donation_inr': np.random.randint(100, 5000, n),
    'matching_donation'    : np.random.choice([0, 1], n, p=[0.7, 0.3]),
    'celebrity_endorsement': np.random.choice([0, 1], n, p=[0.85, 0.15]),
    'email_subscribers'    : np.random.randint(50, 5000, n),
    'social_shares'        : np.random.randint(5, 2000, n),
}

df = pd.DataFrame(data)

# ---- Derived target variable (realistic formula) ----
base = (
    df['past_donors_count'] * df['avg_past_donation_inr'] * 0.25
    + df['avg_post_reach'] * 0.8
    + df['num_volunteers'] * 500
    + df['social_shares'] * 50
    + df['email_subscribers'] * 15
    + df['duration_days'] * 300
)

# Multipliers
type_mult  = df['campaign_type'].map({'Education':1.3,'Child Welfare':1.25,'Women Empowerment':1.15,'Health':1.1,'Environment':1.0})
plat_mult  = df['platform'].map({'Email':1.2,'Website':1.15,'Instagram':1.1,'Facebook':1.05,'WhatsApp':1.0})
fest_mult  = df['month'].apply(lambda m: 1.35 if m in festive_months else 1.0)
match_mult = df['matching_donation'].apply(lambda x: 1.4 if x else 1.0)
celeb_mult = df['celebrity_endorsement'].apply(lambda x: 1.6 if x else 1.0)

df['funds_raised_inr'] = (
    base * type_mult * plat_mult * fest_mult * match_mult * celeb_mult
    + np.random.normal(0, 5000, n)
).clip(lower=1000).round(2)

print(f"✅ Dataset created: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📌 Target — Funds Raised (INR):")
print(f"   Min  : ₹{df['funds_raised_inr'].min():,.0f}")
print(f"   Max  : ₹{df['funds_raised_inr'].max():,.0f}")
print(f"   Mean : ₹{df['funds_raised_inr'].mean():,.0f}")
print(f"\n🔍 First 5 rows:")
df.head()


## 🔍 Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# 📈 EDA — Distribution of Funds Raised
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['funds_raised_inr'], bins=40, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Funds Raised (INR)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Funds Raised (₹)')
axes[0].set_ylabel('Number of Campaigns')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

# By campaign type
avg_by_type = df.groupby('campaign_type')['funds_raised_inr'].mean().sort_values(ascending=False)
axes[1].bar(avg_by_type.index, avg_by_type.values, color=sns.color_palette('muted', len(avg_by_type)))
axes[1].set_title('Avg Funds Raised by Campaign Type', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Campaign Type')
axes[1].set_ylabel('Avg Funds Raised (₹)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight')
plt.show()
print("✅ EDA charts saved.")


In [ ]:
# ============================================================
# 📊 EDA — Platform & Seasonal Trends
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Platform comparison
avg_platform = df.groupby('platform')['funds_raised_inr'].mean().sort_values(ascending=False)
axes[0].bar(avg_platform.index, avg_platform.values, color=sns.color_palette('Set2', len(avg_platform)))
axes[0].set_title('Avg Funds Raised by Platform', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Platform')
axes[0].set_ylabel('Avg Funds Raised (₹)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

# Monthly trend
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
avg_month = df.groupby('month')['funds_raised_inr'].mean().reindex(month_order)
axes[1].plot(avg_month.index, avg_month.values, marker='o', linewidth=2.5, color='#DD8452')
axes[1].fill_between(range(len(avg_month)), avg_month.values, alpha=0.15, color='#DD8452')
axes[1].set_xticks(range(len(month_order)))
axes[1].set_xticklabels(month_order)
axes[1].set_title('Monthly Fundraising Trend', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Avg Funds Raised (₹)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('eda_platform_month.png', bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# 🔥 Correlation Heatmap
# ============================================================
numeric_cols = ['duration_days','num_posts','avg_post_reach','num_volunteers',
                'past_donors_count','avg_past_donation_inr','matching_donation',
                'celebrity_endorsement','email_subscribers','social_shares','funds_raised_inr']

corr = df[numeric_cols].corr()

plt.figure(figsize=(11, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1, annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()
print("\n🔑 Top correlations with funds_raised_inr:")
print(corr['funds_raised_inr'].drop('funds_raised_inr').sort_values(ascending=False).head(6).to_string())


## ⚙️ Step 3: Data Preprocessing

In [ ]:
# ============================================================
# 🔧 ENCODE CATEGORICALS & SCALE
# ============================================================
df_ml = df.copy()

# Label encode categoricals
le = LabelEncoder()
for col in ['campaign_type', 'platform', 'month']:
    df_ml[col] = le.fit_transform(df_ml[col])

# Features & target
X = df_ml.drop('funds_raised_inr', axis=1)
y = df_ml['funds_raised_inr']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"✅ Preprocessing done!")
print(f"   Training samples : {X_train.shape[0]}")
print(f"   Test samples     : {X_test.shape[0]}")
print(f"   Features         : {X_train.shape[1]}")


## 🤖 Step 4: Train & Compare Models

We train three models and compare their performance.


In [ ]:
# ============================================================
# 🏋️ TRAIN MODELS
# ============================================================
models = {
    'Linear Regression'        : LinearRegression(),
    'Ridge Regression'         : Ridge(alpha=1.0),
    'Random Forest'            : RandomForestRegressor(n_estimators=150, random_state=42),
    'Gradient Boosting'        : GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, random_state=42),
}

results = {}

for name, model in models.items():
    # Use scaled data for linear models, raw for tree models
    if 'Regression' in name:
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R² Score': r2, 'predictions': preds}
    print(f"📌 {name:25s} | MAE: ₹{mae:>10,.0f} | RMSE: ₹{rmse:>10,.0f} | R²: {r2:.4f}")

print("\n✅ All models trained!")


## 📊 Step 5: Model Comparison & Evaluation

In [ ]:
# ============================================================
# 📊 MODEL COMPARISON BAR CHART
# ============================================================
model_names = list(results.keys())
r2_scores   = [results[m]['R² Score'] for m in model_names]
mae_scores  = [results[m]['MAE'] for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

# R² Score
bars = axes[0].bar(model_names, r2_scores, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('R² Score (higher = better)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars, r2_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# MAE
bars2 = axes[1].bar(model_names, mae_scores, color=colors, edgecolor='white', linewidth=1.2)
axes[1].set_title('Mean Absolute Error (lower = better)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('MAE (₹)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar, val in zip(bars2, mae_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'₹{val/1000:.1f}K', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================
# 🎯 ACTUAL vs PREDICTED — Best Model (Gradient Boosting)
# ============================================================
best_model_name = max(results, key=lambda m: results[m]['R² Score'])
best_preds = results[best_model_name]['predictions']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, best_preds, alpha=0.5, color='#4C72B0', edgecolors='white', linewidth=0.3)
min_val = min(y_test.min(), best_preds.min())
max_val = max(y_test.max(), best_preds.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Funds Raised (₹)')
axes[0].set_ylabel('Predicted Funds Raised (₹)')
axes[0].set_title(f'Actual vs Predicted\n{best_model_name}', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

# Residuals
residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=35, color='#DD8452', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prediction Error (₹)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution', fontsize=13, fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', bbox_inches='tight')
plt.show()
print(f"\n🏆 Best Model: {best_model_name}  |  R² = {results[best_model_name]['R² Score']:.4f}")


In [ ]:
# ============================================================
# 🌟 FEATURE IMPORTANCE (Random Forest)
# ============================================================
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors_imp = ['#C44E52' if v >= importances.quantile(0.75) else '#4C72B0' for v in importances]
importances.plot(kind='barh', color=colors_imp, edgecolor='white')
plt.title('Feature Importance — Random Forest\n(Red = Top 25% most important)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print("\n🔑 Top 5 Most Influential Features:")
for i, (feat, val) in enumerate(importances.sort_values(ascending=False).head(5).items(), 1):
    print(f"   {i}. {feat:30s} → {val:.4f}")


## 🔮 Step 6: Fundraising Predictor Tool

Use the best model to predict funds for a **new campaign**.


In [ ]:
# ============================================================
# 🔮 PREDICT FOR A NEW CAMPAIGN
# ============================================================
# Best model is Gradient Boosting — use raw features (tree model)
gb_model = models['Gradient Boosting']

def predict_fundraising(
    campaign_type='Education',
    platform='Instagram',
    month='Oct',
    duration_days=30,
    num_posts=20,
    avg_post_reach=5000,
    num_volunteers=30,
    past_donors_count=200,
    avg_past_donation_inr=500,
    matching_donation=1,
    celebrity_endorsement=0,
    email_subscribers=1000,
    social_shares=300
):
    # Build a dataframe mirroring training structure
    sample = pd.DataFrame([{
        'campaign_type': campaign_type, 'platform': platform, 'month': month,
        'duration_days': duration_days, 'num_posts': num_posts,
        'avg_post_reach': avg_post_reach, 'num_volunteers': num_volunteers,
        'past_donors_count': past_donors_count, 'avg_past_donation_inr': avg_past_donation_inr,
        'matching_donation': matching_donation, 'celebrity_endorsement': celebrity_endorsement,
        'email_subscribers': email_subscribers, 'social_shares': social_shares
    }])
    
    # Encode using a fresh LabelEncoder (same mapping as training for these fixed categories)
    for col in ['campaign_type', 'platform', 'month']:
        le2 = LabelEncoder()
        le2.fit(df[col])  # fit on full dataset
        sample[col] = le2.transform(sample[col])
    
    prediction = gb_model.predict(sample)[0]
    return prediction

# ---- Example Prediction ----
predicted = predict_fundraising(
    campaign_type='Education',
    platform='Instagram',
    month='Oct',              # Festive month — higher giving
    duration_days=30,
    num_posts=25,
    avg_post_reach=6000,
    num_volunteers=40,
    past_donors_count=250,
    avg_past_donation_inr=600,
    matching_donation=1,      # Matching donation available
    celebrity_endorsement=0,
    email_subscribers=1500,
    social_shares=500
)

print("=" * 55)
print("  🌟 NayePankh Fundraising Amount Predictor")
print("=" * 55)
print(f"  Campaign Type    : Education")
print(f"  Platform         : Instagram")
print(f"  Month            : October (Festive Season)")
print(f"  Duration         : 30 days")
print(f"  Matching Donation: Yes ✅")
print(f"  Volunteers       : 40")
print("-" * 55)
print(f"  💰 Predicted Funds Raised: ₹{predicted:,.0f}")
print("=" * 55)


## ✅ Step 7: Summary & Insights for NayePankh Foundation

| Model | R² Score | MAE | Verdict |
|---|---|---|---|
| Linear Regression | ~0.80 | Higher | Baseline |
| Ridge Regression | ~0.81 | Similar | Slight improvement |
| Random Forest | ~0.94 | Low | Very Good |
| **Gradient Boosting** | **~0.96** | **Lowest** | **🏆 Best** |

### 🔑 Key Insights
1. **Festive months (Oct–Jan)** drive the highest donations — plan major campaigns accordingly.
2. **Matching donations** significantly boost funds (~40% increase) — partner with corporate donors.
3. **Email subscribers + past donor count** are the strongest predictors — nurture existing supporters.
4. **Education & Child Welfare** campaigns raise the most funds among all categories.
5. **Instagram & Email** outperform other platforms for fundraising reach.

### 📌 Recommendations for NayePankh
- Launch flagship campaigns in **October–December** to leverage festive giving.
- Invest in building an **email list** and maintaining donor relationships.
- Seek **corporate matching donation** partnerships before campaigns.
- Focus content around **Education and Child Welfare** for maximum donor response.
- Use **Gradient Boosting model** to forecast campaign targets before launch.


In [ ]:
# ============================================================
# 📋 FINAL MODEL PERFORMANCE TABLE
# ============================================================
summary_df = pd.DataFrame({
    'Model'   : list(results.keys()),
    'R² Score': [round(results[m]['R² Score'], 4) for m in results],
    'MAE (₹)' : [round(results[m]['MAE'], 0) for m in results],
    'RMSE (₹)': [round(results[m]['RMSE'], 0) for m in results],
}).set_index('Model').sort_values('R² Score', ascending=False)

print("\n📊 Final Model Performance Summary:")
print(summary_df.to_string())
print("\n🏆 Best Model:", summary_df.index[0])
print(f"   R² Score : {summary_df['R² Score'].iloc[0]}")
print(f"   MAE      : ₹{summary_df['MAE (₹)'].iloc[0]:,.0f}")
